<h1 style=\"text-align: center; font-size: 75px;\"> ⚙️ Run Workflow: Agentic Audio RAG </h1>

📘 Project Overview: 
 This notebook demonstrates a modular architecture for answering natural language questions 
 over one or more transcribed audio/video files documents using only local and open-source models (e.g., LLaMA.cpp, OpenAI Whisper).
 The system processes transcript documents chunk-by-chunk and synthesizes a final answer using a multi-step LLM workflow.

# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [ ]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Local application-specific imports
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    display_image,
    get_project_root,
    get_response_from_llm,
    json_schema_from_type,
    log_timing,
    sec_to_timestamp,
    logger,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [ ]:
QUESTION: str = "What are the main issues regarding the product"
DOCS: list
FILE_ID: str
MEMORY: SimpleKVMemory
INPUT_PATH: Path = Path("../data/input")

# Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 11.6 ms, sys: 23.4 ms, total: 35 ms
Wall time: 1.69 s


In [ ]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import base64  # Provides encoding and decoding of binary data
import functools  # Higher-order functions for functional programming
import json  # JSON serialization and deserialization
import logging  # Flexible logging system
import multiprocessing  # Support for spawning processes
import shutil  # High-level file operations
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support

# ─────── Third-Party Packages ───────
import yaml  # YAML parsing and serialization
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
from huggingface_hub import snapshot_download
from huggingface_hub import hf_hub_download
import torch

# ─────── LangChain Core & Community ───────
from langchain.docstore.document import Document  # Core document abstraction
# from langchain.text_splitter import RecursiveCharacterTextSplitter  # Text chunking utility
# from langchain_community.document_loaders import (  # Loaders for various document types
#     CSVLoader,
#     PyPDFLoader,
#     TextLoader,
#     UnstructuredExcelLoader,
#     UnstructuredMarkdownLoader,
#     UnstructuredWordDocumentLoader,
# )
from langchain_community.llms import LlamaCpp  # Integration for local LlamaCpp models

# ─────── LangGraph ───────
# from langgraph.graph import END, START, StateGraph  # Graph components for stateful workflows

from src.agentic_workflow import build_agentic_graph
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state

# Configure Settings

In [6]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
project_root = get_project_root()
MEMORY_PATH: Path = Path("../data/memory")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
# Default model path in case Audio Models fail to download
MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
SAMPLE_MEDIA_PATH = INPUT_PATH / "sample.mp3"
CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Download Sample Audio File

In [ ]:
wav = hf_hub_download(
        repo_id="roneneldan/TEDLIUM_sample",
        filename="TED_0001.wav",
        cache_dir=INPUT_PATH,
)
print("Saved to:", wav)

## Configuration and Secrets Loading

In this section, we load configuration parameters and API keys from separate YAML files. This separation helps maintain security by keeping sensitive information (API keys) separate from configuration settings.

- **config.yaml**: Contains non-sensitive configuration parameters like model sources and URLs
- **secrets.yaml**: Contains sensitive API keys for services like HuggingFace
- *(Optional for Premium users)* Secrets such as API keys for services like HuggingFace can be stored as environment variables for the project and loaded into the notebook (see the project's README file for steps on how to save secrets in Secrets Manager).

In [ ]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

In [ ]:
configure_proxy(config)

## Verify Assets

In [9]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

In [ ]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)
log_asset_status(
    asset_path=MODEL_PATH,
    asset_name="LLM",
)

In [ ]:
assert SAMPLE_MEDIA_PATH.exists(), "Sample media file not found. Please ensure the required asset is correctly configured in your AI Studio project according to the README file."

# Download Model From HuggingFace

This cell imports the snapshot_download function from the huggingface_hub library to download the latest version of the "mispeech/midashenglm-7b" model. It sets a local directory for saving the model (local_model_path), and the download is configured to be resumable in case it is interrupted, with an etag timeout set to 60 seconds.

In [ ]:
GGUF_REPO, GGUF_FILE = {
 #   "MiDaSheng": ("MiSpeech/MiDaShengLM-7B-GGUF", "MiDaShengLM-7B-Q8_0.gguf"),
 #   "Kimi": ("Moonshot-AI/Kimi-Audio-7B-Instruct-GGUF", "Kimi-Audio-7B-Instruct-Q8_0.gguf"),
    "Qwen": ("Qwen/Qwen2_5-Omni-7B-GGUF", "qwen2_5-omni-7b-q8_0.gguf"), 
}["Qwen"]

In [ ]:
%%time
# Download the snapshot directly to the project's models directory
local_model_path = str(get_project_root() / "models" / "audio-language")
gguf_path = os.path.join(local_model_path, GGUF_FILE)

if not os.path.exists(gguf_path):
    print("⬇️  Downloading", GGUF_FILE)
    gguf_path = Path(
        snapshot_download(
            repo_id=GGUF_REPO,
            file_name=GGUF_FILE,
            local_dir=local_model_path,
            resume_download=True,
            etag_timeout=60,
        )
    )

print("✅ Model ready →", gguf_path)


# Whisper Model and Transcript Setup

In [ ]:
import whisper

whisper_model = whisper.load_model("large-v3")
raw_transcript = whisper_model.transcribe(str(SAMPLE_MEDIA_PATH), fp16=False)
full_transcript = raw_transcript["text"].strip()


In [ ]:
FILE_ID = SAMPLE_MEDIA_PATH.name
DOCS = [Document(page_content=full_transcript)]
print(full_transcript[:300] + "...")

# LLM Setup

In [ ]:
%%time

if not os.path.exists(gguf_path):
    model_path = str(MODEL_PATH)
else:
    model_path = str(gguf_path)

llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=-1,                             
    n_batch=512,                                 
    n_ctx=CONTEXT_WINDOW,
    max_tokens=MAX_TOKENS,
    f16_kv=True,
    use_mmap=False,                             
    low_vram=False,                            
    rope_scaling=None,
    temperature=0.0,
    repeat_penalty=1.0,
    streaming=False,
    stop=None,
    seed=42,
    num_threads=multiprocessing.cpu_count(),
    verbose=False                                
)

CPU times: user 1.18 s, sys: 2.68 s, total: 3.86 s
Wall time: 1min 13s


# KV Memory

In [ ]:
MEMORY = SimpleKVMemory(MEMORY_PATH)

# Build and Compile LangGraph

In [ ]:
graph = build_agentic_graph()
compiled_graph = graph.compile()

In [ ]:
png = compiled_graph.get_graph().draw_mermaid_png()
display_image(png)

# Run Agentic Workflow

In [ ]:
%%time

final_graph = compiled_graph.invoke(
        input={
            "docs": DOCS,
            "file_id": FILE_ID,
            "question": QUESTION,
            "input_path": INPUT_PATH, 
            "memory": MEMORY, 
            "llm": llm,
            "messages": [],
        },
    )

# Generated Answer and Snippets

In [ ]:
answer = final_graph.get('answer')
display(Markdown(answer))

In [ ]:
for snippet in final_graph.get('snippets'):
    display(Markdown(snippet))

# Message History

In [ ]:
pretty_json = json.dumps(final_graph.get('messages'), indent=4)
print(pretty_json)

In [20]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).